In [13]:
import joblib # Added import for joblib
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")


In [25]:
import gradio as gr
import yfinance as yf
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib # Added import for joblib

def predict_logic(symbol):
    try:
        # 1. הורדת הנתונים
        df = yf.download(symbol, period="1mo", interval="1d")

        # 2. התיקון הקריטי: ביטול הכותרות הכפולות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        # 3. מחיקת העמודה רק אם היא קיימת (זה ימנע את השגיאה שראית בתמונה)
        if 'Adj Close' in df.columns:
            df = df.drop(columns=['Adj Close'])

        # 4. המשך הלוגיקה של המודל שלך...
        # למשל: features = prepare_features(df)
        # prediction = model.predict(features)

        return "הצלחה!", "כאן יבוא החיזוי" # דוגמה להחזרה

    except Exception as e:
        # זה מה שמציג את השגיאה שראית בתמונה
        return f"Error: {str(e)}", None

# 1. טעינת המודל (החלף את השם לשם הקובץ שלך)
model = joblib.load("/content/drive/MyDrive/פרויקט גמר/xgb_model (1).pkl")
# Removed: model = xgb_model(1).pkl() (Incorrect syntax)
# Removed: model.load_model("my_xgboost_model.json") (Conflicting with joblib load)

def get_features(symbol):
    # משיכת נתונים מספיקים ליצירת Features (למשל חודש אחרון)
  #  symbol = "^GSPC"
    df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
    print("GOOD morning")
    # df = yf.download(symbol, period="1mo", interval="1d")
    if df.empty: return None

    # יצירת ה-Features בדיוק כפי שעשית במחברת (דוגמאות נפוצות):
    df['Returns'] = df['Close'].pct_change()
    df['MA5'] = df['Close'].rolling(window=5).mean()
    df['MA20'] = df['Close'].rolling(window=20).mean()
    df['Volatility'] = df['Returns'].rolling(window=5).std()

    # לקיחת השורה האחרונה ביותר לחיזוי
    latest_features = df.tail(1).drop(columns=['Adj Close']) # התאם לעמודות שלך
    return latest_features, df

def predict_logic(symbol):
    try:
        features_row, full_df = get_features(symbol)

        # הרצת החיזוי (כאן המודל שלך נכנס לפעולה)
        # prediction = model.predict(features_row)[0] # 0 או 1
        # prob = model.predict_proba(features_row)[0] # הסתברות

        # --- דוגמה ויזואלית ללא המודל הטעון ---
        last_price = full_df['Close'].iloc[-1]
        prev_price = full_df['Close'].iloc[-2]
        trend = "UP 🚀" if last_price > prev_price else "DOWN 📉"
        conf = {"עליה": 0.65, "ירידה": 0.35} # כאן יבוא ה-prob מהמודל

        # יצירת גרף היסטורי קטן
        fig, ax = plt.subplots(figsize=(6, 3))
        full_df['Close'].tail(15).plot(ax=ax, color='blue', lw=2)
        ax.set_title(f"Price History: {symbol}")
        plt.tight_layout()

        return trend, conf, fig

    except Exception as e:
        return f"Error: {str(e)}", None, None

# 2. בניית ממשק המשתמש (UI)
with gr.Blocks(title="S&P 500 Predictor") as demo:
    gr.Markdown("# 🤖 חוזה מגמת מניות - AI Project")
    gr.Markdown("האפליקציה מושכת נתונים מ-Yahoo Finance ומריצה מודל XGBoost לחיזוי המגמה.")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל ^GSPC או AAPL)", value="^GSPC")
        btn = gr.Button("נתח עכשיו", variant="primary")

    with gr.Row():
        with gr.Column():
            out_label = gr.Textbox(label="תחזית המודל (עליה/ירידה)")
            out_prob = gr.Label(label="הסתברות")
        with gr.Column():
            out_plot = gr.Plot(label="גרף מחירים אחרון")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[out_label, out_prob, out_plot])

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2d60faeabb5579f856.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [16]:
import yfinance as yf

symbol = "^GSPC"
df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
print(df.head(10))

/tmp/ipykernel_445/644755225.py:4: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(symbol, period="1mo", interval="1d", multi_level_index=False)
[*********************100%***********************]  1 of 1 completed

                  Close         High          Low         Open      Volume
Date                                                                      
2026-02-17  6843.220215  6866.990234  6775.500000  6819.859863  5418480000
2026-02-18  6881.310059  6909.120117  6849.660156  6855.479980  5098160000
2026-02-19  6861.890137  6879.120117  6833.060059  6861.339844  5151690000
2026-02-20  6909.509766  6915.859863  6836.330078  6843.259766  5432480000
2026-02-23  6837.750000  6916.959961  6819.819824  6901.250000  5638350000
2026-02-24  6890.069824  6899.169922  6815.430176  6837.370117  5266090000
2026-02-25  6946.129883  6952.509766  6915.149902  6915.149902  5328060000
2026-02-26  6908.859863  6947.250000  6859.729980  6944.740234  5889550000
2026-02-27  6878.879883  6882.959961  6831.740234  6856.540039  6665660000
2026-03-02  6881.620117  6901.009766  6796.850098  6824.359863  6079080000


In [26]:
import gradio as gr
import yfinance as yf
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

# --- 1. טעינת המודל (השאר את זה ככה אם עוד לא טענת קובץ) ---
# model = joblib.load("your_model.pkl")

def predict_logic(symbol):
    try:
        # א. הורדת נתונים (בלי להסתמך על הגדרות yfinance)
        df = yf.download(symbol, period="1mo", interval="1d")

        if df.empty:
            return "לא נמצאו נתונים עבור הסימול הזה", None, None

        # ב. התיקון הקריטי: "ניקוי" כותרות כפולות (Multi-index)
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)
            # אם יש עמודה שנקראת 'Ticker', נוריד אותה
            if 'Ticker' in df.columns:
                df = df.drop(columns=['Ticker'])

        # ג. מחיקת עמודות בשיטה בטוחה שלא גורמת לשגיאה
        # במקום drop רגיל, אנחנו פשוט בוחרים רק את העמודות שאנחנו רוצים
        available_cols = df.columns.tolist()
        needed_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
        # לוקחים רק מה שבאמת קיים בטבלה מתוך הרשימה שלנו
        final_df = df[[col for col in needed_cols if col in available_cols]].copy()

        # ד. לוגיקת החיזוי (כאן תכניס את ה-Preprocessing שלך)
        last_price = final_df['Close'].iloc[-1]
        prev_price = final_df['Close'].iloc[-2]

        # דוגמה לתוצאה ויזואלית
        trend = "עליה (UP) 🚀" if last_price > prev_price else "ירידה (DOWN) 📉"
        prob = {"עליה": 0.7, "ירידה": 0.3} # כאן יבוא ה-predict_proba מהמודל שלך

        # יצירת גרף
        fig, ax = plt.subplots(figsize=(6, 3))
        final_df['Close'].tail(15).plot(ax=ax, color='green' if last_price > prev_price else 'red')
        ax.set_title(f"Price: {symbol}")
        plt.tight_layout()

        return trend, prob, fig

    except Exception as e:
        # אם יש שגיאה, נציג אותה בצורה ברורה
        return f"שגיאה בתהליך: {str(e)}", None, None

# --- 2. ממשק ה-Gradio ---
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🤖 חוזה מגמת S&P 500")

    with gr.Row():
        input_symbol = gr.Textbox(label="הכנס סימול (למשל ^GSPC)", value="^GSPC")
        btn = gr.Button("בצע חיזוי", variant="primary")

    with gr.Row():
        with gr.Column():
            res_text = gr.Textbox(label="תחזית")
            res_label = gr.Label(label="הסתברות")
        with gr.Column():
            res_plot = gr.Plot(label="גרף מחירים אחרון")

    btn.click(fn=predict_logic, inputs=input_symbol, outputs=[res_text, res_label, res_plot])

demo.launch()

/tmp/ipykernel_445/734871063.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://08b07779e130c85acf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
